# 09 - Evaluacion CT por estudio

Esta fase complementa la clasificacion CT por slice. MosMedData asigna la etiqueta de severidad al estudio completo, no a cada corte individual; por eso aqui se agregan las probabilidades de todos los slices del mismo `study_id` para obtener una prediccion unica por estudio.

El notebook no reentrena modelos. Lee las predicciones ya guardadas en `results/classification/ct`, ejecuta el script reproducible `scripts/build_ct_study_level_analysis.py` y muestra tablas, figuras e interpretacion.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'results' / 'classification' / 'ct_study_level'
FIGURES_DIR = OUTPUT_DIR / 'figures'
PYTHON_BIN = PROJECT_ROOT / '.conda' / 'bin' / 'python'
PYTHON_BIN = str(PYTHON_BIN if PYTHON_BIN.exists() else sys.executable)

PROJECT_ROOT

## Regenerar evaluacion

El script genera una evaluacion por estudio usando varias estrategias de agregacion:

- `mean_probability`: promedio de probabilidades de todos los slices;
- `confidence_weighted_mean`: promedio ponderado por confianza;
- `max_probability`: maximo score por clase;
- `top3_mean_probability`: promedio de los tres slices mas fuertes por clase;
- `top20pct_mean_probability`: promedio del 20% de slices mas fuertes;
- `majority_vote`: voto mayoritario de predicciones por slice.

El objetivo es comprobar si el rendimiento CT mejora al evaluar en la misma unidad en la que MosMedData define la etiqueta: el estudio completo.

In [ ]:
RUN_BUILD = True

if RUN_BUILD:
    subprocess.run(
        [PYTHON_BIN, str(PROJECT_ROOT / 'scripts' / 'build_ct_study_level_analysis.py')],
        cwd=PROJECT_ROOT,
        check=True,
    )

## Cargar tablas

Cargamos las metricas por modelo y estrategia de agregacion. Cada fila `study` representa una evaluacion sobre 167 estudios de test; las filas `slice` mantienen la evaluacion original sobre 4155 slices.

In [ ]:
metrics_df = pd.read_csv(OUTPUT_DIR / 'ct_study_level_metrics.csv')
study_predictions_df = pd.read_csv(OUTPUT_DIR / 'ct_study_level_predictions.csv')
reports_df = pd.read_csv(OUTPUT_DIR / 'ct_study_level_classification_reports.csv')

print({'metric_rows': len(metrics_df), 'study_predictions': len(study_predictions_df), 'report_rows': len(reports_df)})
display(metrics_df.head())

## Mejores resultados por estudio

Ordenamos las estrategias por F1-macro porque CT esta desbalanceado y el F1-macro da importancia similar a las clases minoritarias.

In [ ]:
cols = ['experiment', 'unit', 'aggregation', 'n_samples', 'accuracy', 'f1_macro', 'f1_weighted', 'auc_roc_macro']
study_metrics = metrics_df[metrics_df['unit'] == 'study'].sort_values(['f1_macro', 'accuracy'], ascending=False)
display(study_metrics[cols].head(15).reset_index(drop=True))

## Comparacion slice vs estudio

Aqui comparamos la evaluacion original por slice con la evaluacion por estudio usando `mean_probability`, que es la agregacion mas sencilla y defendible para la memoria.

In [ ]:
slice_metrics = metrics_df[metrics_df['unit'] == 'slice'][['experiment', 'accuracy', 'f1_macro', 'auc_roc_macro']]
mean_study_metrics = metrics_df[(metrics_df['unit'] == 'study') & (metrics_df['aggregation'] == 'mean_probability')][['experiment', 'accuracy', 'f1_macro', 'auc_roc_macro']]
comparison_df = slice_metrics.merge(mean_study_metrics, on='experiment', suffixes=('_slice', '_study'))
comparison_df['delta_f1_macro'] = comparison_df['f1_macro_study'] - comparison_df['f1_macro_slice']
comparison_df['delta_accuracy'] = comparison_df['accuracy_study'] - comparison_df['accuracy_slice']
display(comparison_df.sort_values('delta_f1_macro', ascending=False).reset_index(drop=True))

## Figuras generadas

Las figuras ayudan a documentar visualmente el cambio de unidad de evaluacion y la matriz de confusion del mejor modelo por estudio.

In [ ]:
for figure_name in [
    'ct_slice_vs_study_f1_macro.png',
    'ct_study_level_aggregation_comparison.png',
    'ct_study_level_best_confusion_matrix.png',
]:
    figure_path = FIGURES_DIR / figure_name
    print(figure_path)
    display(Image(filename=str(figure_path)))

## Informe generado

El script crea un resumen en Markdown con la lectura metodologica principal. Esta lectura se puede trasladar a la seccion de resultados o discusion de la memoria.

In [ ]:
display(Markdown((OUTPUT_DIR / 'ct_study_level_summary.md').read_text()))

## Conclusion metodologica

La evaluacion por estudio es mas coherente con MosMedData porque la etiqueta CT-0/CT-1/CT-2/CT-3+ pertenece al volumen completo. Al agregar los slices de un mismo paciente se reduce el peso de cortes aislados poco informativos. Esta fase no invalida la evaluacion por slice, sino que la complementa y ofrece una lectura mas defendible para CT.